In [1]:
import pandas as pd
from utils.utils import setup_model, pull_spectral_images, save_model
from sklearn.model_selection import train_test_split
from net.generator import SpectralImageDataGenerator
from net.net_def import AutoencoderHyperModel
from kerastuner.tuners import Hyperband
import json
import subprocess
import webbrowser
import time


# Load the TensorBoard notebook extension
%load_ext tensorboard

In [2]:
# Model Configuration
MODEL_NAME = 'hyper_3d_autoencoder'
INPUT_FILES = "hyspecnet-11k/patches"
INPUT_FILE_TYPE = 'tif'
BATCH_SIZE = 32
EPOCHS = 10
USE_CALLBACKS = ['tf_logs', 'early_stopping']

In [3]:
# Load the spectral images
spectral_images, input_shape = pull_spectral_images(INPUT_FILES, type=INPUT_FILE_TYPE)
print(f"Loaded {len(spectral_images)} spectral images with input shape {input_shape}.")

Pulling spectral images from .TIF files
Loaded 1222 spectral images with input shape (128, 128, 224, 1).


In [4]:
# Split the data into training and validation sets
image_paths_train, image_paths_val = train_test_split(spectral_images, test_size=0.2, random_state=42)
train_gen = SpectralImageDataGenerator(image_paths_train, batch_size=BATCH_SIZE)
val_gen = SpectralImageDataGenerator(image_paths_val, batch_size=BATCH_SIZE)
print(f"Inputs set to {input_shape} with {len(image_paths_train)} training images and {len(image_paths_val)} validation images.")

Creating generator with 977 images. Batch size: 32 Shuffle: False
Creating generator with 245 images. Batch size: 32 Shuffle: False
Inputs set to (128, 128, 224, 1) with 977 training images and 245 validation images.


In [5]:
model_dir, callbacks = setup_model(MODEL_NAME, use_callbacks=USE_CALLBACKS)

hypermodel = AutoencoderHyperModel(input_shape=input_shape)  # Adjusted to use the dynamic input_shape

tuner = Hyperband(
    hypermodel,
    objective='val_mae',
    directory=model_dir,
    project_name='tuning',
    max_consecutive_failed_trials=50
)

# Start TensorBoard in browser
tb_process = subprocess.Popen(['tensorboard', '--logdir', model_dir, '--bind_all'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)
webbrowser.open_new("http://localhost:6006/")


# Start the search process
tuner.search(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

Trial 33 Complete [00h 00m 09s]

Best val_mae So Far: 0.6951819062232971
Total elapsed time: 01h 32m 01s

Search: Running Trial #34

Value             |Best Value So Far |Hyperparameter
1                 |2                 |conv_layers
32                |64                |filters_0
192               |256               |dense_units
1                 |2                 |convT_layers
64                |48                |filtersT_0
adam              |rmsprop           |optimizer
0.0077473         |0.0004772         |lr_adam
64                |32                |filters_1
48                |48                |filtersT_1
64                |16                |filtersT_2
0.002075          |0.00096949        |lr_rmsprop
0.006872          |0.00035228        |lr_sgd
0                 |0.3               |momentum_sgd
64                |64                |filters_2
2                 |2                 |tuner/epochs
0                 |0                 |tuner/initial_epoch
4                 |4    

: 

In [ ]:
# After search, retrieve the best model
best_model = tuner.get_best_models(num_models=1)[0]
best_hyperparameters = tuner.get_best_hyperparameters()[0]
best_hyperparameters.values

In [ ]:
# After tuning, save the best hyperparameters
saved_model = save_model(best_model, MODEL_NAME, model_dir)  
best_hyperparameters = tuner.get_best_hyperparameters()[0].get_config()
hp_filename = f"{model_dir}/best_hyperparameters.json"

with open(hp_filename, 'w') as f:
    json.dump(best_hyperparameters, f)

print(f"Best hyperparameters saved to {hp_filename}")
print(json.dumps(best_hyperparameters, indent=4))